<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/hybrid-rag-search-pipeline/blob/main/hybrid_search_rag_from_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Install Required Libraries and Packages and Import It

In [4]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 57.3 MB/s eta 0:00:00


In [15]:
import os
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter

###Chunking and Splitting

####Flat chunking (Chunking using Chunk size and Overlap Size) and Get Contents

In [9]:
def get_files_content(pdf_path):
  doc = fitz.open(pdf_path)
  text = ""

  for page in doc:
    text += page.get_text()+"\n"

  return text


In [11]:
def chunk_text(text, chunk_size=1000, overlap_size= 200):
  chunks = []

  for i in range(0, len(text), chunk_size - overlap_size):
    chunk = text[i:i+ chunk_size]
    chunks.append(chunk)

  return chunks


In [18]:
def recursive_text_splitter(text, chunk_size=1000, overlap_size= 200):
   splitter = RecursiveCharacterTextSplitter(
       chunk_size = chunk_size,
       chunk_overlap = overlap_size,
       separators = [
           "\n",
           "\n\n",
           ". ",
           " ",
           ""
       ]
   )

   return splitter.split_text(text)

#### Metadata Chunking

In [22]:
def extract_pages(pdf_path):
  doc = fitz.open(pdf_path)
  pages = []

  for page_num, page in enumerate(doc, start=1):
    pages.append({
        "page": page_num,
        "text": page.get_text()
    })

  return pages

In [25]:
def recersive_chunk_pages(pages, chunk_size = 1000, overlap_size = 200):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = overlap_size,
      separators = [
           "\n",
           "\n\n",
           ". ",
           " ",
           ""
       ]
  )

  chunks = []
  id = 1

  for page in pages:
     page_chunks = splitter.split_text(page["text"])

     for chunk in page_chunks:
        chunks.append({
            "chunk_id": id,
            "text": chunk,
            "start_page": page["page"]
        })
        id += 1

  return chunks

####Overlap + Metadata chunking

In [31]:
import fitz

def extract_pdf_with_page_map(pdf_path):
    doc = fitz.open(pdf_path)

    full_text = ""
    page_map = []

    current_pos = 0

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        start = current_pos
        full_text += text + "\n"
        end = len(full_text)

        page_map.append({
            "page": page_num,
            "start": start,
            "end": end
        })

        current_pos = end

    return full_text, page_map

In [32]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk_text = text[i:i + chunk_size]

        chunks.append({
            "text": chunk_text,
            "start": i,
            "end": i + len(chunk_text)
        })

    return chunks

In [33]:
def add_page_numbers(chunks, page_map):
    final_chunks = []

    for chunk in chunks:
        start_page = None
        end_page = None

        for page in page_map:
            # check overlap
            if chunk["start"] <= page["end"] and chunk["end"] >= page["start"]:

                if start_page is None:
                    start_page = page["page"]

                end_page = page["page"]

        final_chunks.append({
            "text": chunk["text"],
            "start_page": start_page,
            "end_page": end_page
        })

    return final_chunks

In [35]:
full_text, page_map = extract_pdf_with_page_map('/content/Marcus-Aurelius-Meditations.pdf')
chunks = chunk_text(full_text, 1000, 200)
final_output = add_page_numbers(chunks, page_map)